# Environment Setup

In [13]:
# 🔧 Core Python libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from datetime import datetime,timedelta
import requests
import os
import time
import io
import zipfile
from math import radians, cos, sin, asin, sqrt

# 🧰 Sklearn libraries
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from xgboost import XGBClassifier
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


# Data Acqisition:

In [2]:
# 1. Define what we want to download
symbol = "BTCUSDT"
interval = "15m"
base_url = f"https://data.binance.vision/data/spot/monthly/klines/{symbol}/{interval}/"

# Let's get the last 12 months of data for a robust MVP dataset
months_to_fetch = 12
end_date = datetime.now()
file_list = []

# Generate the list of filenames (ZIP files, not CSV)
for i in range(months_to_fetch):
    # Calculate the target month
    target_date = end_date - timedelta(days=30*i)
    year = target_date.year
    month = str(target_date.month).zfill(2)
    
    # Binance ZIP URL format: BTCUSDT-15m-2026-07.zip
    filename = f"{symbol}-{interval}-{year}-{month}.zip"
    url = base_url + filename
    file_list.append((filename, url))

print(f" Preparing to download {len(file_list)} monthly zip files...")

# 2. Download, extract, and combine the data
all_dataframes = []

for filename, url in file_list:
    try:
        print(f"  Fetching: {filename}...", end=" ")
        response = requests.get(url)
        response.raise_for_status()
        
        # Extract the CSV from the ZIP file
        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
            # Get the CSV filename from inside the zip
            csv_filename = zip_file.namelist()[0]
            
            # Read the CSV from inside the zip
            with zip_file.open(csv_filename) as csv_file:
                df = pd.read_csv(csv_file, header=None)
        
        # Name the columns (Binance doesn't provide headers in these raw files)
        df.columns = [
            'open_time', 'open', 'high', 'low', 'close', 'volume',
            'close_time', 'quote_asset_volume', 'number_of_trades',
            'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore'
        ]
        
        all_dataframes.append(df)
        print("✅")
        
    except requests.exceptions.HTTPError:
        # Some very old months might not exist, we just skip them
        print("️ Not found, skipping.")
    except zipfile.BadZipFile:
        print("⚠️ Corrupted zip, skipping.")

# 3. Combine all monthly dataframes into one giant DataFrame
if all_dataframes:
    print("\n Combining data...")
    df_raw = pd.concat(all_dataframes, ignore_index=True)

    # 4. Clean and Sort (CRITICAL for Time Series)
    print("⏳ Cleaning and sorting by time...")
    df_raw['open_time'] = pd.to_datetime(df_raw['open_time'], unit='ms')

    # Convert numeric columns to float
    numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'taker_buy_base_asset_volume']
    df_raw[numeric_cols] = df_raw[numeric_cols].astype(float)

    # Sort chronologically and remove any exact duplicate rows (just in case)
    df_raw = df_raw.sort_values('open_time').drop_duplicates(subset=['open_time']).reset_index(drop=True)

    # 5. Final Report
    print("\n" + "="*50)
    print("✅ DATA ACQUISITION COMPLETE")
    print("="*50)
    print(f"Total Rows:      {len(df_raw):,}")
    print(f"Date Range:      {df_raw['open_time'].min()} to {df_raw['open_time'].max()}")
    print(f"Memory Usage:    {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"Columns:         {list(df_raw.columns)}")
    print("="*50)

    # Show the last 5 rows to verify it's up to date
    print("\n👀 Latest 5 candles:")
    print(df_raw[['open_time', 'open', 'high', 'low', 'close', 'volume']].tail())
else:
    print("\n❌ ERROR: No data was downloaded. Check your internet connection or the date range.")

 Preparing to download 12 monthly zip files...
  Fetching: BTCUSDT-15m-2026-08.zip... ️ Not found, skipping.
  Fetching: BTCUSDT-15m-2026-07.zip... ✅
  Fetching: BTCUSDT-15m-2026-06.zip... ✅
  Fetching: BTCUSDT-15m-2026-05.zip... ✅
  Fetching: BTCUSDT-15m-2026-04.zip... ✅
  Fetching: BTCUSDT-15m-2026-03.zip... ✅
  Fetching: BTCUSDT-15m-2026-02.zip... ✅
  Fetching: BTCUSDT-15m-2026-01.zip... ✅
  Fetching: BTCUSDT-15m-2025-12.zip... ✅
  Fetching: BTCUSDT-15m-2025-11.zip... ✅
  Fetching: BTCUSDT-15m-2025-10.zip... ✅
  Fetching: BTCUSDT-15m-2025-10.zip... ✅

 Combining data...
⏳ Cleaning and sorting by time...

✅ DATA ACQUISITION COMPLETE
Total Rows:      29,184
Date Range:      57719-04-07 00:00:00 to 58551-07-23 14:00:00
Memory Usage:    2.67 MB
Columns:         ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore']

👀 Latest 5 candles:
                 open_t

# Exploratory Data Analysis:

In [3]:
print("="*60)
print("🔄 STEP 2: ETL - General Structure, Missing Values, Duplicates")
print("="*60)

# Assuming 'df_raw' is the dataframe we got from Step 1 (Data Acquiring)
# If you are running this in a new session, you would load it here. 
# For now, we assume df_raw is in memory.

# ============================================================================
# 1. GENERAL STRUCTURE & DATA TYPES
# ============================================================================
print("\n 1. GENERAL STRUCTURE")
print("-" * 60)
print(f"DataFrame Shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
print("\nColumn Names:")
print(df_raw.columns.tolist())
print("\nData Types:")
print(df_raw.dtypes)

# ============================================================================
# 2. MISSING VALUES CHECK
# ============================================================================
print("\n 2. MISSING VALUES")
print("-" * 60)
missing_values = df_raw.isnull().sum()
missing_pct = (missing_values / len(df_raw)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing %': missing_pct.round(2)
})

# Filter to show only columns with missing values
missing_summary = missing_df[missing_df['Missing Count'] > 0]
if missing_summary.empty:
    print("✅ No missing values found in any column.")
else:
    print("⚠️ Missing values detected:")
    print(missing_summary)

# ============================================================================
# 3. DUPLICATE CHECK
# ============================================================================
print("\n 3. DUPLICATE CHECK")
print("-" * 60)
# Check for exact duplicate rows
exact_duplicates = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {exact_duplicates}")

# Check for duplicate timestamps (critical for time series)
time_duplicates = df_raw.duplicated(subset=['open_time']).sum()
print(f"Duplicate timestamps: {time_duplicates}")

if time_duplicates > 0:
    print("\n⚠️ Sample of duplicate timestamps:")
    print(df_raw[df_raw.duplicated(subset=['open_time'], keep=False)][['open_time', 'open', 'close']].head())

# ============================================================================
# 4. BASIC DATA TYPE VALIDATION (Transform)
# ============================================================================
print("\n 4. DATA TYPE VALIDATION")
print("-" * 60)
# Ensure open_time is datetime
if not pd.api.types.is_datetime64_any_dtype(df_raw['open_time']):
    print("⚠️ 'open_time' is not datetime. Converting...")
    df_raw['open_time'] = pd.to_datetime(df_raw['open_time'], unit='ms')
else:
    print("✅ 'open_time' is correctly formatted as datetime.")

# Ensure price and volume columns are numeric
numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'taker_buy_base_asset_volume']
for col in numeric_cols:
    if not pd.api.types.is_numeric_dtype(df_raw[col]):
        print(f"⚠️ '{col}' is not numeric. Converting to float...")
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

print("✅ All core columns validated for correct data types.")

# ============================================================================
# 5. LOAD (Save to disk for reproducibility)
# ============================================================================
print("\n 5. SAVE PROCESSED RAW DATA (LOAD)")
print("-" * 60)

# Create a data directory if it doesn't exist
os.makedirs('data/raw', exist_ok=True)

# Save as Parquet (faster and smaller than CSV, preserves data types perfectly)
output_path = 'data/raw/btcusdt_15m_etl.parquet'
df_raw.to_parquet(output_path, index=False)
print(f"✅ Data successfully saved to: {output_path}")
print(f"   File size: {os.path.getsize(output_path) / 1024:.2f} KB")

print("\n" + "="*60)
print("✅ STEP 2 COMPLETE: ETL Finished.")
print("="*60)

🔄 STEP 2: ETL - General Structure, Missing Values, Duplicates

 1. GENERAL STRUCTURE
------------------------------------------------------------
DataFrame Shape: 29,184 rows, 12 columns

Column Names:
['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore']

Data Types:
open_time                       datetime64[ms]
open                                   float64
high                                   float64
low                                    float64
close                                  float64
volume                                 float64
close_time                               int64
quote_asset_volume                     float64
number_of_trades                         int64
taker_buy_base_asset_volume            float64
taker_buy_quote_asset_volume           float64
ignore                                   int64
dtype: object

 2. MISSING VALUES
--

# Data Cleaning

In [4]:
print("="*60)
print("🧹 STEP 3: Data Cleaning (Strictly No Data Leakage)")
print("="*60)

# 1. Load the ETL data from Step 2
# (Adjust the extension if you saved as CSV instead of Parquet)
try:
    df_clean = pd.read_parquet('data/raw/btcusdt_15m_etl.parquet')
except FileNotFoundError:
    df_clean = pd.read_csv('data/raw/btcusdt_15m_etl.csv', parse_dates=['open_time'])

print(f"Loaded {len(df_clean):,} rows for cleaning.")

# ============================================================================
# 2. ENSURE CHRONOLOGICAL ORDER
# ============================================================================
print("\n 1. ENSURING CHRONOLOGICAL ORDER")
print("-" * 60)
# Time series MUST be sorted oldest to newest before any cleaning
df_clean = df_clean.sort_values('open_time').reset_index(drop=True)
print("✅ Data sorted by 'open_time'.")

# ============================================================================
# 3. DROP DUPLICATES (No Leakage)
# ============================================================================
print("\n 2. DROPPING DUPLICATES")
print("-" * 60)
initial_rows = len(df_clean)
# Drop rows with the exact same timestamp. Keep the first one.
df_clean = df_clean.drop_duplicates(subset=['open_time'], keep='first')
print(f"Dropped {initial_rows - len(df_clean)} duplicate timestamps.")

# ============================================================================
# 4. HANDLE MISSING VALUES (NO LEAKAGE)
# ============================================================================
print("\n 3. HANDLING MISSING VALUES (Forward Fill)")
print("-" * 60)

# Define the core columns that cannot have missing values
core_cols = ['open', 'high', 'low', 'close', 'volume', 'taker_buy_base_asset_volume']

# --- THE NO LEAKAGE RULE ---
# We DO NOT use df_clean['close'].mean() to fill missing values. 
# That would use future data to fix past data (Leakage!).
# Instead, we use Forward Fill (ffill). If a price is missing, we assume 
# it stayed at the last known price. This only looks backward in time.

missing_before = df_clean[core_cols].isnull().sum().sum()

# Forward fill missing values
df_clean[core_cols] = df_clean[core_cols].ffill()

# Edge case: If the very first row has NaNs, ffill can't fix it (there is no past).
# We safely drop those specific rows.
df_clean = df_clean.dropna(subset=core_cols)

missing_after = df_clean[core_cols].isnull().sum().sum()

print(f"Missing values before: {missing_before}")
print(f"Missing values after:  {missing_after}")
print("✅ Used Forward Fill (ffill) to prevent data leakage.")

# ============================================================================
# 5. HANDLE OUTLIERS (Wait for Step 4/5)
# ============================================================================
print("\n 4. OUTLIER HANDLING")
print("-" * 60)
print("⚠️ Skipping outlier clipping/dropping for now.")
print("Reason: In crypto, massive spikes (flash crashes/pumps) are real market events.")
print("Dropping them now would create artificial gaps in the time series.")
print("We will let the ML model handle them, or handle them AFTER the train/test split.")

# ============================================================================
# 6. SAVE CLEANED DATA
# ============================================================================
print("\n 5. SAVE CLEANED DATA")
print("-" * 60)

os.makedirs('data/processed', exist_ok=True)

try:
    output_path = 'data/processed/btcusdt_15m_clean.parquet'
    df_clean.to_parquet(output_path, index=False)
    print(f"✅ Cleaned data saved to: {output_path}")
except ImportError:
    output_path = 'data/processed/btcusdt_15m_clean.csv'
    df_clean.to_csv(output_path, index=False)
    print(f"✅ Cleaned data saved to: {output_path}")

print(f"Final dataset shape: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns")

print("\n" + "="*60)
print("✅ STEP 3 COMPLETE: Data Cleaning Finished (No Leakage).")
print("="*60)

🧹 STEP 3: Data Cleaning (Strictly No Data Leakage)
Loaded 29,184 rows for cleaning.

 1. ENSURING CHRONOLOGICAL ORDER
------------------------------------------------------------
✅ Data sorted by 'open_time'.

 2. DROPPING DUPLICATES
------------------------------------------------------------
Dropped 0 duplicate timestamps.

 3. HANDLING MISSING VALUES (Forward Fill)
------------------------------------------------------------
Missing values before: 0
Missing values after:  0
✅ Used Forward Fill (ffill) to prevent data leakage.

 4. OUTLIER HANDLING
------------------------------------------------------------
⚠️ Skipping outlier clipping/dropping for now.
Reason: In crypto, massive spikes (flash crashes/pumps) are real market events.
Dropping them now would create artificial gaps in the time series.
We will let the ML model handle them, or handle them AFTER the train/test split.

 5. SAVE CLEANED DATA
------------------------------------------------------------
✅ Cleaned data saved to

# Chronological(based on time) Train/Test/Eval Split

In [5]:
print("="*60)
print("✂️ STEP 4: Chronological Data Splitting (No Leakage)")
print("="*60)

# 1. Load the cleaned data from Step 3
try:
    df_clean = pd.read_parquet('data/processed/btcusdt_15m_clean.parquet')
except FileNotFoundError:
    df_clean = pd.read_csv('data/processed/btcusdt_15m_clean.csv', parse_dates=['open_time'])

print(f"Loaded {len(df_clean):,} rows for splitting.")

# ============================================================================
# 2. DEFINE SPLIT RATIOS
# ============================================================================
print("\n 1. DEFINING SPLIT RATIOS")
print("-" * 60)
# Standard time-series split: 80% Train, 10% Validation, 10% Test
train_ratio = 0.80
val_ratio = 0.10
test_ratio = 0.10

total_rows = len(df_clean)

# Calculate the exact row indices for the splits
train_end_idx = int(total_rows * train_ratio)
val_end_idx = int(total_rows * (train_ratio + val_ratio))

print(f"Total rows: {total_rows:,}")
print(f"Train rows: {train_end_idx:,} (0 to {train_end_idx})")
print(f"Val rows:   {val_end_idx - train_end_idx:,} ({train_end_idx} to {val_end_idx})")
print(f"Test rows:  {total_rows - val_end_idx:,} ({val_end_idx} to end)")

# ============================================================================
# 3. PERFORM THE CHRONOLOGICAL SPLIT
# ============================================================================
print("\n 2. SPLITTING DATA")
print("-" * 60)

# Slice the dataframe strictly by row index (which is sorted by time)
df_train = df_clean.iloc[:train_end_idx].copy()
df_val = df_clean.iloc[train_end_idx:val_end_idx].copy()
df_test = df_clean.iloc[val_end_idx:].copy()

# Reset indices so they start at 0 for each set
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print("✅ Data successfully sliced chronologically.")

# ============================================================================
# 4. SANITY CHECK (Crucial for Time Series)
# ============================================================================
print("\n 3. SANITY CHECK (Verifying No Overlap & Correct Order)")
print("-" * 60)

def print_set_info(name, df):
    print(f"\n--- {name} SET ---")
    print(f"Rows: {len(df):,}")
    print(f"Start Date: {df['open_time'].min()}")
    print(f"End Date:   {df['open_time'].max()}")
    print(f"Avg Price:  ${df['close'].mean():,.2f}")

print_set_info("TRAIN", df_train)
print_set_info("VALIDATION", df_val)
print_set_info("TEST", df_test)

# Verify no time overlap
assert df_train['open_time'].max() < df_val['open_time'].min(), " ERROR: Train and Val overlap!"
assert df_val['open_time'].max() < df_test['open_time'].min(), "❌ ERROR: Val and Test overlap!"
print("\n✅ Time overlap check passed. No leakage detected.")

# ============================================================================
# 5. SAVE SPLITS TO DISK
# ============================================================================
print("\n 4. SAVING SPLITS")
print("-" * 60)

os.makedirs('data/splits', exist_ok=True)

try:
    df_train.to_parquet('data/splits/train.parquet', index=False)
    df_val.to_parquet('data/splits/val.parquet', index=False)
    df_test.to_parquet('data/splits/test.parquet', index=False)
    print("✅ Splits saved as Parquet files in 'data/splits/'.")
except ImportError:
    df_train.to_csv('data/splits/train.csv', index=False)
    df_val.to_csv('data/splits/val.csv', index=False)
    df_test.to_csv('data/splits/test.csv', index=False)
    print("✅ Splits saved as CSV files in 'data/splits/'.")

print("\n" + "="*60)
print("✅ STEP 4 COMPLETE: Data Splitting Finished.")
print("="*60)

✂️ STEP 4: Chronological Data Splitting (No Leakage)
Loaded 29,184 rows for splitting.

 1. DEFINING SPLIT RATIOS
------------------------------------------------------------
Total rows: 29,184
Train rows: 23,347 (0 to 23347)
Val rows:   2,918 (23347 to 26265)
Test rows:  2,919 (26265 to end)

 2. SPLITTING DATA
------------------------------------------------------------
✅ Data successfully sliced chronologically.

 3. SANITY CHECK (Verifying No Overlap & Correct Order)
------------------------------------------------------------

--- TRAIN SET ---
Rows: 23,347
Start Date: 57719-04-07 00:00:00
End Date:   58385-02-01 12:00:00
Avg Price:  $85,285.65

--- VALIDATION SET ---
Rows: 2,918
Start Date: 58385-02-11 22:00:00
End Date:   58468-04-22 08:00:00
Avg Price:  $63,151.52

--- TEST SET ---
Rows: 2,919
Start Date: 58468-05-02 18:00:00
End Date:   58551-07-23 14:00:00
Avg Price:  $63,862.75

✅ Time overlap check passed. No leakage detected.

 4. SAVING SPLITS
----------------------------

# Data Engneering:

In [ ]:
print("="*60)
print("⚙️ STEP 5: Feature Engineering & Target Creation")
print("="*60)

# 1. Load the split data from Step 4
print("\n 1. LOADING SPLITS")
print("-" * 60)
df_train = pd.read_parquet('data/splits/train.parquet')
df_val = pd.read_parquet('data/splits/val.parquet')
df_test = pd.read_parquet('data/splits/test.parquet')

# Remember original lengths so we can split them back later
train_len = len(df_train)
val_len = len(df_val)

# Temporarily combine them to calculate rolling features without leakage
df_combined = pd.concat([df_train, df_val, df_test], ignore_index=True)
print(f"Combined dataset for feature calculation: {len(df_combined):,} rows")

# ============================================================================
# 2. FEATURE ENGINEERING (The "Lenses")
# ============================================================================
print("\n 2. CALCULATING FEATURES")
print("-" * 60)

# --- A. Returns (Speed) ---
df_combined['return_1'] = df_combined['close'].pct_change(1)
df_combined['return_3'] = df_combined['close'].pct_change(3)

# --- B. Volatility (Chaos) ---
# 10-period rolling standard deviation of the 1-step return
df_combined['volatility_10'] = df_combined['return_1'].rolling(window=10).std()

# --- C. Trend (Direction) ---
# Exponential Moving Averages (gives more weight to recent prices)
df_combined['ema_20'] = df_combined['close'].ewm(span=20, adjust=False).mean()
df_combined['ema_50'] = df_combined['close'].ewm(span=50, adjust=False).mean()

# Distance from EMA (helps model see if price is overextended)
df_combined['dist_from_ema_20'] = (df_combined['close'] - df_combined['ema_20']) / df_combined['ema_20']

# --- D. Momentum (RSI) ---
# Pure Pandas implementation of 14-period RSI
delta = df_combined['close'].diff()
gain = delta.where(delta > 0, 0.0).ewm(alpha=1/14, min_periods=14).mean()
loss = (-delta.where(delta < 0, 0.0)).ewm(alpha=1/14, min_periods=14).mean()
rs = gain / loss
df_combined['rsi_14'] = 100 - (100 / (1 + rs))

# --- E. Volume (Conviction) ---
# Current volume compared to the 20-period average volume
df_combined['volume_sma_20'] = df_combined['volume'].rolling(window=20).mean()
df_combined['volume_ratio'] = df_combined['volume'] / df_combined['volume_sma_20']

print("✅ Features calculated successfully.")

# ============================================================================
# 3. TARGET ENGINEERING (The "Answer Key")
# ============================================================================
print("\n 3. CREATING TARGET (30-Minute Direction)")
print("-" * 60)

# We want to predict the direction 30 minutes from now. 
# Since each candle is 15 mins, we look 2 candles into the future.
df_combined['future_close'] = df_combined['close'].shift(-2)
df_combined['future_return'] = (df_combined['future_close'] - df_combined['close']) / df_combined['close']

# Define the threshold for Bullish/Bearish (0.2% or 0.002)
threshold = 0.002 

def assign_target(ret):
    if pd.isna(ret): return np.nan
    if ret >= threshold: return 1   # Bullish
    elif ret <= -threshold: return -1 # Bearish
    else: return 0                  # Neutral

df_combined['target'] = df_combined['future_return'].apply(assign_target)

# Check target distribution
target_counts = df_combined['target'].value_counts().to_dict()
print(f"Target distribution (before dropping NaNs): {target_counts}")

# ============================================================================
# 4. RE-SPLIT AND CLEAN
# ============================================================================
print("\n 4. RE-SPLITTING AND CLEANING")
print("-" * 60)

# Slice back into the original sets
df_train_eng = df_combined.iloc[:train_len].copy()
df_val_eng = df_combined.iloc[train_len:train_len + val_len].copy()
df_test_eng = df_combined.iloc[train_len + val_len:].copy()

# Drop rows with NaN values. 
# This removes the first ~50 rows (where rolling windows hadn't filled yet) 
# and the last 2 rows (where the future target doesn't exist yet).
df_train_eng = df_train_eng.dropna().reset_index(drop=True)
df_val_eng = df_val_eng.dropna().reset_index(drop=True)
df_test_eng = df_test_eng.dropna().reset_index(drop=True)

print(f"Train rows after cleaning: {len(df_train_eng):,}")
print(f"Val rows after cleaning:   {len(df_val_eng):,}")
print(f"Test rows after cleaning:  {len(df_test_eng):,}")

# ============================================================================
# 5. SAVE ENGINEERED DATA
# ============================================================================
print("\n 5. SAVING ENGINEERED DATA")
print("-" * 60)

os.makedirs('data/features', exist_ok=True)

try:
    df_train_eng.to_parquet('data/features/train_eng.parquet', index=False)
    df_val_eng.to_parquet('data/features/val_eng.parquet', index=False)
    df_test_eng.to_parquet('data/features/test_eng.parquet', index=False)
    print("✅ Engineered features saved to 'data/features/'.")
except ImportError:
    df_train_eng.to_csv('data/features/train_eng.csv', index=False)
    df_val_eng.to_csv('data/features/val_eng.csv', index=False)
    df_test_eng.to_csv('data/features/test_eng.csv', index=False)
    print("✅ Engineered features saved as CSV to 'data/features/'.")

print("\n" + "="*60)
print("✅ STEP 5 COMPLETE: Feature Engineering Finished.")
print("="*60)

⚙️ STEP 5: Feature Engineering & Target Creation

 1. LOADING SPLITS
------------------------------------------------------------
Combined dataset for feature calculation: 29,184 rows

 2. CALCULATING FEATURES
------------------------------------------------------------
✅ Features calculated successfully.

 3. CREATING TARGET (30-Minute Direction)
------------------------------------------------------------
Target distribution (before dropping NaNs): {0.0: 17784, -1.0: 5767, 1.0: 5631}

 4. RE-SPLITTING AND CLEANING
------------------------------------------------------------
Train rows after cleaning: 23,328
Val rows after cleaning:   2,918
Test rows after cleaning:  2,917

 5. SAVING ENGINEERED DATA
------------------------------------------------------------
✅ Engineered features saved to 'data/features/'.

✅ STEP 5 COMPLETE: Feature Engineering Finished.
Ready for Step 6: Build pipeline on training data.


# Building the Pipline on the Training Data Only:

In [9]:
print("="*60)
print("🔧 STEP 6: Build Pipeline on Training Data")
print("="*60)

# ============================================================================
# 1. LOAD ENGINEERED TRAINING DATA
# ============================================================================
print("\n 1. LOADING TRAINING DATA")
print("-" * 60)
try:
    df_train = pd.read_parquet('data/features/train_eng.parquet')
except FileNotFoundError:
    df_train = pd.read_csv('data/features/train_eng.csv')

print(f"Loaded {len(df_train):,} rows for pipeline construction.")

# ============================================================================
# 2. SEPARATE FEATURES (X) AND TARGET (y)
# ============================================================================
print("\n 2. SEPARATING FEATURES AND TARGET")
print("-" * 60)

# Columns to DROP: 
# - 'open_time': Not a predictive feature (just an index)
# - 'target': This is what we are trying to predict (the answer key)
# - 'future_close', 'future_return': These are future values used to create the target. 
#   If we leave them in X, the model will cheat (massive data leakage!).
cols_to_drop = ['open_time', 'target', 'future_close', 'future_return']

# Ensure we only drop columns that actually exist
cols_to_drop = [col for col in cols_to_drop if col in df_train.columns]

X_train = df_train.drop(columns=cols_to_drop)
y_train = df_train['target']

print(f"Feature matrix (X_train) shape: {X_train.shape}")
print(f"Target vector (y_train) shape: {y_train.shape}")
print(f"\nFeatures to be used:\n{list(X_train.columns)}")

# ============================================================================
# 3. DEFINE THE PREPROCESSING STEPS
# ============================================================================
print("\n 3. DEFINING PREPROCESSING STEPS")
print("-" * 60)

# Even though tree-based models (like XGBoost) don't strictly need scaling, 
# it is a best practice to include it in the pipeline. 
# It ensures that if we swap in a model that DOES need scaling (like Logistic Regression 
# or SVM) in Step 7, the pipeline is already ready for it.

# Step A: Handle any remaining tiny NaNs (though we dropped them in Step 5, 
# it's a safety net for the pipeline)
imputer = SimpleImputer(strategy='median')

# Step B: Scale the features to have mean=0 and variance=1
scaler = StandardScaler()

# Combine them into a ColumnTransformer
# This applies the imputer and scaler to ALL numeric columns in X_train
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', imputer),
            ('scaler', scaler)
        ]), X_train.columns) # Apply to all current columns
    ],
    remainder='drop' # Drop any non-numeric columns that might sneak in
)

print("✅ Preprocessor defined (Imputer + StandardScaler).")

# ============================================================================
# 4. BUILD THE FULL PIPELINE (With a Placeholder Model)
# ============================================================================
print("\n 4. BUILDING THE FULL PIPELINE")
print("-" * 60)

# We will define the actual models in Step 7, but we need to build the 
# pipeline structure now. Let's use a simple baseline model (Random Forest) 
# to ensure the pipeline works end-to-end.
from sklearn.ensemble import RandomForestClassifier

# Create the pipeline
# The names 'preprocessor' and 'model' can be anything, but 'model' is standard
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

print("✅ Full Pipeline constructed successfully.")
print(f"Pipeline steps: {pipeline.named_steps.keys()}")

# ============================================================================
# 5. FIT THE PIPELINE ON TRAINING DATA ONLY
# ============================================================================
print("\n 5. FITTING PIPELINE ON TRAINING DATA")
print("-" * 60)
print("Fitting... (This learns the scaling parameters and trains the baseline model)")

# This single line does:
# 1. preprocessor.fit_transform(X_train)
# 2. model.fit(X_train_processed, y_train)
pipeline.fit(X_train, y_train)

print("✅ Pipeline successfully fitted on training data.")

# ============================================================================
# 6. SAVE THE PIPELINE
# ============================================================================
print("\n 6. SAVING THE PIPELINE")
print("-" * 60)

import joblib

os.makedirs('models', exist_ok=True)
pipeline_path = 'models/baseline_pipeline.joblib'

joblib.dump(pipeline, pipeline_path)
print(f"✅ Pipeline saved to: {pipeline_path}")
print(f"   File size: {os.path.getsize(pipeline_path) / 1024:.2f} KB")

print("\n" + "="*60)
print("✅ STEP 6 COMPLETE: Pipeline Built and Fitted.")
print("Ready for Step 7: Define Models.")
print("="*60)

🔧 STEP 6: Build Pipeline on Training Data

 1. LOADING TRAINING DATA
------------------------------------------------------------
Loaded 23,328 rows for pipeline construction.

 2. SEPARATING FEATURES AND TARGET
------------------------------------------------------------
Feature matrix (X_train) shape: (23328, 20)
Target vector (y_train) shape: (23328,)

Features to be used:
['open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore', 'return_1', 'return_3', 'volatility_10', 'ema_20', 'ema_50', 'dist_from_ema_20', 'rsi_14', 'volume_sma_20', 'volume_ratio']

 3. DEFINING PREPROCESSING STEPS
------------------------------------------------------------
✅ Preprocessor defined (Imputer + StandardScaler).

 4. BUILDING THE FULL PIPELINE
------------------------------------------------------------
✅ Full Pipeline constructed successfully.
Pipeline steps: dict_keys(['preprocessor', '

In [12]:
print("="*60)
print("🔄 STEP 6.5: Explicit Data Transformation")
print("="*60)

# ============================================================================
# 1. LOAD THE FITTED PIPELINE AND DATA
# ============================================================================
print("\n 1. LOADING PIPELINE AND RAW FEATURE DATA")
print("-" * 60)

# Load the pipeline we just fitted and saved in Previous step
pipeline = joblib.load('models/baseline_pipeline.joblib')
print("✅ Pipeline loaded from 'models/baseline_pipeline.joblib'")

# Extract ONLY the preprocessor from the pipeline
# (We don't want to transform the data through the Random Forest model yet)
preprocessor = pipeline.named_steps['preprocessor']
print("✅ Extracted 'preprocessor' step from pipeline.")

# Load the engineered datasets
df_train = pd.read_parquet('data/features/train_eng.parquet')
df_val = pd.read_parquet('data/features/val_eng.parquet')
df_test = pd.read_parquet('data/features/test_eng.parquet')

# ============================================================================
# 2. SEPARATE X AND y (Dropping Leaky Columns)
# ============================================================================
print("\n 2. SEPARATING X AND y")
print("-" * 60)

cols_to_drop = ['open_time', 'target', 'future_close', 'future_return']
cols_to_drop = [col for col in cols_to_drop if col in df_train.columns]

X_train_raw = df_train.drop(columns=cols_to_drop)
y_train = df_train['target']

X_val_raw = df_val.drop(columns=cols_to_drop)
y_val = df_val['target']

X_test_raw = df_test.drop(columns=cols_to_drop)
y_test = df_test['target']

print(f"X_train shape: {X_train_raw.shape}")
print(f"X_val shape:   {X_val_raw.shape}")
print(f"X_test shape:  {X_test_raw.shape}")

# ============================================================================
# 3. EXPLICIT TRANSFORMATION
# ============================================================================
print("\n 3. TRANSFORMING DATA (Applying Imputer & Scaler)")
print("-" * 60)

# Transform the data using the preprocessor fitted ONLY on the training data
X_train_proc = preprocessor.transform(X_train_raw)
X_val_proc = preprocessor.transform(X_val_raw)
X_test_proc = preprocessor.transform(X_test_raw)

# The preprocessor outputs a NumPy array. Let's convert it back to a DataFrame 
# so we keep our feature names for later analysis (like SHAP values).
X_train_proc = pd.DataFrame(X_train_proc, columns=X_train_raw.columns)
X_val_proc = pd.DataFrame(X_val_proc, columns=X_val_raw.columns)
X_test_proc = pd.DataFrame(X_test_proc, columns=X_test_raw.columns)

print("✅ Data successfully transformed.")

# Quick sanity check: The mean of the training data should be very close to 0
print("\n👀 Sanity Check (Mean of first 3 features in X_train):")
print(X_train_proc.iloc[:, :3].mean().round(4))

# ============================================================================
# 4. SAVE PROCESSED DATA
# ============================================================================
print("\n 4. SAVING PROCESSED DATA")
print("-" * 60)

os.makedirs('data/processed_features', exist_ok=True)

# Save X and y separately. This is standard practice for ML modeling.
X_train_proc.to_parquet('data/processed_features/X_train_proc.parquet', index=False)
y_train.to_frame(name='target').to_parquet('data/processed_features/y_train.parquet', index=False)

X_val_proc.to_parquet('data/processed_features/X_val_proc.parquet', index=False)
y_val.to_frame(name='target').to_parquet('data/processed_features/y_val.parquet', index=False)

X_test_proc.to_parquet('data/processed_features/X_test_proc.parquet', index=False)
y_test.to_frame(name='target').to_parquet('data/processed_features/y_test.parquet', index=False)

print("✅ All processed X and y datasets saved to 'data/processed_features/'.")

print("\n" + "="*60)
print("✅ STEP 6.5 COMPLETE: Data Explicitly Transformed.")
print("="*60)

🔄 STEP 6.5: Explicit Data Transformation

 1. LOADING PIPELINE AND RAW FEATURE DATA
------------------------------------------------------------
✅ Pipeline loaded from 'models/baseline_pipeline.joblib'
✅ Extracted 'preprocessor' step from pipeline.

 2. SEPARATING X AND y
------------------------------------------------------------
X_train shape: (23328, 20)
X_val shape:   (2918, 20)
X_test shape:  (2917, 20)

 3. TRANSFORMING DATA (Applying Imputer & Scaler)
------------------------------------------------------------
✅ Data successfully transformed.

👀 Sanity Check (Mean of first 3 features in X_train):
open    0.0
high   -0.0
low     0.0
dtype: float64

 4. SAVING PROCESSED DATA
------------------------------------------------------------
✅ All processed X and y datasets saved to 'data/processed_features/'.

✅ STEP 6.5 COMPLETE: Data Explicitly Transformed.


# Defining Model:

In [14]:
print("="*60)
print("🤖 STEP 7: Define Models")
print("="*60)

# ============================================================================
# 1. DEFINE THE MODEL DICTIONARY
# ============================================================================
print("\n 1. INITIALIZING MODELS")
print("-" * 60)

# We store models in a dictionary so we can loop through them later in Step 8 & 9.
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,          # Needs more iterations to converge on financial data
        class_weight='balanced',# Crucial: penalizes the model for ignoring minority classes (Bull/Bear)
        random_state=42,        # Ensures reproducible results
        n_jobs=-1               # Uses all CPU cores
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,       # Number of trees in the forest
        max_depth=10,           # Prevents trees from growing too deep and overfitting
        class_weight='balanced',# Crucial for imbalanced trading targets
        random_state=42,
        n_jobs=-1
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=200,       # Number of boosting rounds
        max_depth=6,            # Depth of each tree
        learning_rate=0.05,     # Shrinkage parameter (smaller = slower learning, less overfitting)
        objective='multi:softprob', # Required for 3-class classification (Bull, Neutral, Bear)
        eval_metric='mlogloss', # Metric to optimize during training
        use_label_encoder=False,# Suppresses a common XGBoost warning
        random_state=42,
        n_jobs=-1
    )
}

print("✅ Models initialized successfully.")

# ============================================================================
# 2. VERIFY MODEL CONFIGURATIONS
# ============================================================================
print("\n 2. MODEL CONFIGURATIONS")
print("-" * 60)

for name, model in models.items():
    print(f"--- {name} ---")
    # Print the most important hyperparameters for quick verification
    if name == "Logistic Regression":
        print(f"  Max Iter: {model.max_iter}")
        print(f"  Class Weight: {model.class_weight}")
    elif name == "Random Forest":
        print(f"  Estimators: {model.n_estimators}")
        print(f"  Max Depth: {model.max_depth}")
        print(f"  Class Weight: {model.class_weight}")
    elif name == "XGBoost":
        print(f"  Estimators: {model.n_estimators}")
        print(f"  Max Depth: {model.max_depth}")
        print(f"  Learning Rate: {model.learning_rate}")
        print(f"  Objective: {model.objective}")
    print()

# ============================================================================
# 3. LOAD PROCESSED DATA (Sanity Check)
# ============================================================================
print("\n 3. VERIFYING DATA SHAPES FOR MODELING")
print("-" * 60)

# Load the explicitly transformed data from Step 6.5
X_train = pd.read_parquet('data/processed_features/X_train_proc.parquet')
y_train = pd.read_parquet('data/processed_features/y_train.parquet')['target']

X_val = pd.read_parquet('data/processed_features/X_val_proc.parquet')
y_val = pd.read_parquet('data/processed_features/y_val_proc.parquet')['target']

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape} | y_val shape: {y_val.shape}")

# Check target distribution to confirm class imbalance
print("\n👀 Target Distribution in Training Set:")
print(y_train.value_counts().sort_index())
print("(-1 = Bearish, 0 = Neutral, 1 = Bullish)")

print("\n" + "="*60)
print("✅ STEP 7 COMPLETE: Models Defined.")
print("Ready for Step 8: Training and Evaluation (Looping through models).")
print("="*60)

🤖 STEP 7: Define Models

 1. INITIALIZING MODELS
------------------------------------------------------------
✅ Models initialized successfully.

 2. MODEL CONFIGURATIONS
------------------------------------------------------------
--- Logistic Regression ---
  Max Iter: 1000
  Class Weight: balanced

--- Random Forest ---
  Estimators: 200
  Max Depth: 10
  Class Weight: balanced

--- XGBoost ---
  Estimators: 200
  Max Depth: 6
  Learning Rate: 0.05
  Objective: multi:softprob


 3. VERIFYING DATA SHAPES FOR MODELING
------------------------------------------------------------


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed_features/y_val_proc.parquet'